# Static Scraping Walkthrough: Quotes to Scrape

This walkthrough introduces static web scraping with `requests` and `BeautifulSoup` using `quotes.toscrape.com`, a small practice site designed for repeated records.

By the end, you should understand how to:

- check `robots.txt` as a first access signal
- fetch raw HTML
- save raw HTML before parsing
- inspect page structure
- use CSS selectors
- extract repeated quote records
- resolve relative URLs
- separate quote, link, and tag tables
- save processed tables

# Imports and Shared Setup


## Packages Used in This Notebook

This walkthrough uses:

- `requests` to fetch the raw HTML page
- `BeautifulSoup` to parse and search the HTML
- `pandas` to store extracted records as tables
- `RobotFileParser` to inspect `robots.txt` as a first access signal
- `urlparse` and `urljoin` to work with URLs
- `Path` to create output folders and file paths

The key division of labor is: `requests` fetches, `BeautifulSoup` parses, `pandas` tabulates.


In [2]:
# Path helps create output folders and file paths.
from pathlib import Path
# pprint makes long strings or nested objects easier to inspect during teaching.
from pprint import pprint
# urljoin resolves relative links; urlparse splits a URL into scheme/domain/path parts.
from urllib.parse import urljoin, urlparse
# pandas turns extracted rows into tables.
import pandas as pd

# requests fetches the raw HTML from the server.
import requests
# RobotFileParser reads robots.txt rules as a first access signal.
from urllib.robotparser import RobotFileParser
# BeautifulSoup parses HTML and lets us search it with selectors.
from bs4 import BeautifulSoup


In [3]:
# All examples in this notebook write into the same teaching data folder.
outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"

# mkdir(..., parents=True, exist_ok=True) creates folders and does not fail if they already exist.
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# Clean Practice Page: Quotes to Scrape

Many scraping tasks involve repeated records: cards, list items, products, posts, search results, or course rows.

`quotes.toscrape.com` is useful because each quote appears in a repeated `.quote` block. That makes the record-container pattern easy to see before working with messier pages.


In [ ]:
URL = "https://quotes.toscrape.com/"
USER_AGENT = "methodsNET-VLOP-course/1.0_static_scraping_walkthrough"


The `User-Agent` identifies the script. It should be meaningful and honest.

# Check robots.txt

`robots.txt` is a plain-text file that many websites publish at the top level of a domain, for example `https://quotes.toscrape.com/robots.txt`. It gives automated clients, including scrapers and search-engine crawlers, instructions about which paths they may or may not request.

We check it before scraping because it is a basic access signal from the site owner. It helps us avoid collecting from paths the site has explicitly marked as disallowed for automated access.

`robots.txt` is not the same as full legal or ethical permission. It does not answer every question about copyright, privacy, terms of service, server load, or whether the data should be collected for a particular research purpose. It is a first check, not the whole decision.

In [ ]:
parsed = urlparse(URL)
robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
# i.e., https://quotes.toscrape.com/robots.txt

rp = RobotFileParser()
rp.set_url(robots_url)
rp.read()

allowed = rp.can_fetch(USER_AGENT, URL)

print("robots.txt URL:", robots_url)
print("Can fetch classroom URL?", allowed)


robots.txt URL: https://quotes.toscrape.com/robots.txt
Can fetch classroom URL? True


::: {.callout-question}
If `robots.txt` allows this URL, what other questions should we still ask before collecting and saving data from the page?
:::

# Fetch the Page

`requests.get()` retrieves the raw HTML returned by the server. It does not
click, scroll, or execute JavaScript.

In [6]:
# requests.get() sends an HTTP GET request. For APIs, params become query-string fields; for web pages, it fetches HTML.
response = requests.get(URL, headers={"User-Agent": USER_AGENT}, timeout=30)
# raise_for_status() turns HTTP error responses (such as 403, 404, or 429) into visible Python errors.
response.raise_for_status()

html = response.text

print("HTTP status:", response.status_code)
print("Characters of HTML:", len(html))
print(html[:500])


HTTP status: 200
Characters of HTML: 11021
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div cla


# Save Raw HTML

Save raw HTML before parsing. This lets us rerun extraction code without
contacting the site again and preserves evidence of what the scraper saw.

In [7]:
raw_html_path = raw_dir / "notebook_quotes_raw.html"
raw_html_path.write_text(html, encoding=response.encoding or "utf-8")

print("Saved raw HTML:", raw_html_path)


Saved raw HTML: data/raw/notebook_quotes_raw.html


## What Happens Between `requests` and BeautifulSoup?

The `requests` package gives us `response.text`: a long string containing the HTML returned by the server.

At that stage, Python has not yet extracted quotes, authors, links, or tags. It only has page source text.

`BeautifulSoup(html, "html.parser")` changes the task: it parses that string into a searchable tree. After that, we can ask questions such as:

- Which elements have class `.quote`?
- Which links have an `href` attribute?
- What visible text is inside `.author`?
- Which URL is stored inside a link's `href` attribute?

Fetching and parsing are separate steps. Saving the raw HTML before parsing preserves evidence of what the scraper actually saw.


# Parse and Inspect HTML

In [8]:
# BeautifulSoup turns HTML text into a searchable parse tree.
soup = BeautifulSoup(html, "html.parser")

# prettify() prints nested HTML with indentation, which makes the page structure easier to inspect.
print(soup.prettify()[:1500])


<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Quotes to Scrape
  </title>
  <link href="/static/bootstrap.min.css" rel="stylesheet"/>
  <link href="/static/main.css" rel="stylesheet"/>
 </head>
 <body>
  <div class="container">
   <div class="row header-box">
    <div class="col-md-8">
     <h1>
      <a href="/" style="text-decoration: none">
       Quotes to Scrape
      </a>
     </h1>
    </div>
    <div class="col-md-4">
     <p>
      <a href="/login">
       Login
      </a>
     </p>
    </div>
   </div>
   <div class="row">
    <div class="col-md-8">
     <div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
      <span class="text" itemprop="text">
       “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
      </span>
      <span>
       by
       <small class="author" itemprop="author">
        Albert Einstein
       </small>
       <a href="/author/Albert

Quick sanity checks:

In [9]:
# get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
print("Page title:", soup.title.get_text(strip=True))
# select() returns a list of all elements matching a CSS selector.
# a[href] means every <a> link element that has an href attribute.
print("Number of links:", len(soup.select("a[href]")))
print("Number of quote blocks:", len(soup.select(".quote")))


Page title: Quotes to Scrape
Number of links: 55
Number of quote blocks: 10


Open the same page in a browser and inspect one quote. Connect the HTML
structure to the selectors `.quote`, `.text`, `.author`, and `.tags`.

# Inspect One Record Before Looping

A good scraper starts with one record. If one record is wrong, the loop will
only repeat the mistake.

In [1]:
# select_one() returns the first element matching a CSS selector, or None if nothing matches.
first_quote = soup.select_one(".quote")

# select() returns all elements matching the CSS selector, or None if nothing matches. select()[2] returns the second element, etc. 
second_quote = soup.select(".quote")[1]

# prettify() prints nested HTML with indentation, which makes the page structure easier to inspect.
print(first_quote.prettify())

NameError: name 'soup' is not defined

Select fields inside that one record:

In [ ]:
# select_one() returns the first element matching a CSS selector, or None if nothing matches.
print("Quote text element:", first_quote.select_one(".text"))
print("Author element:", first_quote.select_one(".author"))
# select() returns a list of all elements matching a CSS selector.
print("Tag link elements:", first_quote.select(".tags a.tag"))
print("Author profile link:", first_quote.select_one("span a[href]"))


## Why Inspect One Record Before Looping?

Scraping usually fails in small ways before it fails in large ways: the selector is too broad, too narrow, or points to the wrong level of the HTML.

That is why the walkthrough first inspects one `.quote` block. Once one record is understood, the loop repeats the same extraction for every quote block.

The key methodological decision is the record container. Here, `.quote` means: one row in the output table represents one quote block on the page.


# Extract Repeated Records

Each `.quote` block is one repeated record container.

In [ ]:
rows = []

# select() returns a list of all elements matching a CSS selector.
for quote_block in soup.select(".quote"):
    # select_one() returns the first element matching a CSS selector, or None if nothing matches.
    text = quote_block.select_one(".text")
    author = quote_block.select_one(".author")
    tag_links = quote_block.select(".tags a.tag")
    author_link = quote_block.select_one("span a[href]")

    row = {
        # get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
        "quote": text.get_text(" ", strip=True) if text else None,
        "author": author.get_text(" ", strip=True) if author else None,
        # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
        "author_href_raw": author_link.get("href") if author_link else None,
        "author_href_absolute": (
            # urljoin() turns a relative URL such as /page/2/ into a full absolute URL using the page URL as context.
            urljoin(URL, author_link.get("href")) if author_link else None
        ),
        "tags": "|".join(tag.get_text(" ", strip=True) for tag in tag_links),
        "tag_count": len(tag_links),
    }

    rows.append(row)

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
quotes_df = pd.DataFrame(rows)
quotes_df.head()


## Why Extract Links Separately?

Links are often analytically different from visible text.

A link has:

- visible anchor text, such as `about`
- an `href` attribute, such as `/author/Albert-Einstein`
- a role on the page, such as author profile, tag page, or pagination

The script classifies links because not every link is part of the same data object. Navigation links, author links, tag links, and next-page links mean different things.


::: {.callout-important}
The selector decides the unit of analysis. Here, `.quote` means one row is one
quote block.
:::

# Extract and Classify Links

All links are `<a>` elements, but not every `<a>` necessarily points somewhere. The selector `a[href]` means: select every `<a>` element that has an `href` attribute.

In plain language, `soup.select("a[href]")` asks BeautifulSoup for all link elements with a URL-like destination we can read with `.get("href")`.

In [ ]:
links = []

# select() returns a list of all elements matching a CSS selector.
# a[href] keeps only <a> elements that include an href attribute.
for link in soup.select("a[href]"):
    # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
    href_raw = link.get("href")
    # urljoin() turns a relative URL such as /page/2/ into a full absolute URL using the page URL as context.
    href_absolute = urljoin(URL, href_raw)

    if href_raw.startswith("/author/"):
        link_type = "author_profile"
    elif href_raw.startswith("/tag/"):
        link_type = "tag_page"
    elif href_raw.startswith("/page/"):
        link_type = "pagination"
    else:
        link_type = "other"

    links.append(
        {
            # get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
            "link_text": link.get_text(" ", strip=True),
            "href_raw": href_raw,
            "href_absolute": href_absolute,
            "link_type": link_type,
        }
    )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
links_df = pd.DataFrame(links)
links_df.head(10)


## Why Make a Separate Tag Table?

A quote can have multiple tags. That creates a design choice.

One table can store tags as a single combined string, for example `life|truth|books`. That is simple and easy to save as CSV.

A separate long table stores one row per quote-tag pair. That is better for counting tags, filtering by tags, or joining tags to other data.

This is not just a Python choice; it changes the shape of the dataset.


# Tags as a Separate Table

Sometimes repeated fields should become their own table.

In [ ]:
tag_rows = []

# select() returns a list of all elements matching a CSS selector.
for quote_number, quote_block in enumerate(soup.select(".quote"), start=1):
    # select_one() returns the first element matching a CSS selector, or None if nothing matches.
    text = quote_block.select_one(".text")
    # get_text(" ", strip=True) extracts visible text, joins internal whitespace with spaces, and removes leading/trailing whitespace.
    quote_text = text.get_text(" ", strip=True) if text else None

    for tag_link in quote_block.select(".tags a.tag"):
        tag_rows.append(
            {
                "quote_number": quote_number,
                "quote": quote_text,
                "tag": tag_link.get_text(" ", strip=True),
                # .get() safely asks for a dictionary key or HTML attribute; if it is missing, Python returns None instead of crashing.
                "tag_url": urljoin(URL, tag_link.get("href")),
            }
        )

# pd.DataFrame(...) turns a list of row dictionaries into a rectangular table.
tags_df = pd.DataFrame(tag_rows)
tags_df.head()


# Save Processed Tables

In [ ]:
quotes_path = processed_dir / "notebook_quotes.csv"
links_path = processed_dir / "notebook_quote_links.csv"
tags_path = processed_dir / "notebook_quote_tags.csv"

# to_csv(..., index=False) saves a DataFrame as a CSV file without adding pandas row numbers as a fake column.
quotes_df.to_csv(quotes_path, index=False)
links_df.to_csv(links_path, index=False)
tags_df.to_csv(tags_path, index=False)

print(quotes_path)
print(links_path)
print(tags_path)


# Warm-Up Exercise: Extend the Quotes Scraper

The walkthrough above extracts the first page of quotes. For this warm-up, extend the scraper so it collects something new instead of repeating the same fields.

Work individually or in pairs:

1. Use `links_df` to find the pagination link for the next page.
2. Fetch and parse that next page.
3. Extract quote records from page 2 using the same `.quote` record container.
4. Add two new columns: `page` and `author_slug`.
5. Combine page 1 and page 2 into one table.
6. Save a short provenance note explaining the start URL, next-page URL, selectors, and date.

Hint: an author profile URL such as `/author/Albert-Einstein` can be turned into the slug `Albert-Einstein` by splitting the URL path on `/` and keeping the last part.


In [ ]:
# Exercise scaffold: find the next-page URL.
# This cell starts from links_df, which the walkthrough already created.

pagination_links = links_df.loc[links_df["link_type"] == "pagination", "href_absolute"]

# TODO: replace None with the first URL in pagination_links.
next_page_url = None

print("Candidate pagination links:")
print(pagination_links.to_list())
print("Selected next page URL:", next_page_url)


In [ ]:
# Exercise scaffold: fetch and parse page 2.
# Run this after setting next_page_url above.

if next_page_url is None:
    print("Set next_page_url before running this cell.")
else:
    page2_response = requests.get(next_page_url, headers={"User-Agent": USER_AGENT}, timeout=30)
    page2_response.raise_for_status()
    page2_soup = BeautifulSoup(page2_response.text, "html.parser")

    print("Page 2 title:", page2_soup.title.get_text(strip=True))
    print("Page 2 quote blocks:", len(page2_soup.select(".quote")))


In [ ]:
# Exercise scaffold: extract page 2 records and combine them with page 1.
# Add an author_slug column. The exact extraction code is for you to complete.

page2_rows = []

if "page2_soup" not in globals():
    print("Create page2_soup before running this cell.")
else:
    for quote_block in page2_soup.select(".quote"):
        text = quote_block.select_one(".text")
        author = quote_block.select_one(".author")
        author_link = quote_block.select_one("span a[href]")
        author_href = author_link.get("href") if author_link else None

        # TODO: derive author_slug from author_href.
        author_slug = None

        page2_rows.append(
            {
                "page": 2,
                "quote": text.get_text(" ", strip=True) if text else None,
                "author": author.get_text(" ", strip=True) if author else None,
                "author_href_raw": author_href,
                "author_href_absolute": urljoin(URL, author_href) if author_href else None,
                "author_slug": author_slug,
            }
        )

    page1_for_combining = quotes_df.copy()
    page1_for_combining["page"] = 1
    page1_for_combining["author_slug"] = None

    page2_df = pd.DataFrame(page2_rows)
    combined_quotes_df = pd.concat([page1_for_combining, page2_df], ignore_index=True, sort=False)
    print("Combined rows:", len(combined_quotes_df))
    combined_quotes_df.head()


In [ ]:
# Exercise scaffold: write a short provenance note.
# TODO: set collection_date and revise the note after you complete the exercise.

collection_date = None

if next_page_url is None or collection_date is None:
    print("Set next_page_url and collection_date before saving the provenance note.")
else:
    provenance_note = f"""
Source start URL: {URL}
Next page URL: {next_page_url}
Collection date: {collection_date}
Selectors used: .quote, .text, .author, span a[href], a[href]
Collection note: page 1 came from the walkthrough; page 2 was added in the warm-up exercise.
""".strip()

    provenance_path = processed_dir / "notebook_quotes_provenance.txt"
    provenance_path.write_text(provenance_note, encoding="utf-8")

    print(provenance_path)
    print(provenance_note)


# Key Takeaway

Scraping is not just "getting what is on a page".

It involves choices about access, selectors, units of analysis, missing data, raw evidence, and documentation.
